In [28]:
import pandas as pd



In [29]:
def load_csv(file):
    try:
        return pd.read_csv(
            file,
            engine="python",
            on_bad_lines="skip",
            encoding="utf-8"
        )
    except:
        return pd.read_csv(
            file,
            engine="python",
            on_bad_lines="skip",
            encoding="latin1"
        )

orders = load_csv("orders_dataset.csv")
order_items = load_csv("order_items_dataset.csv")
payments = load_csv("order_payments_dataset.csv")
reviews = load_csv("order_reviews_dataset.csv")
products = load_csv("products_dataset.csv")
customers = load_csv("customers_dataset.csv")
sellers = load_csv("sellers_dataset.csv")

##Merge Datasets

In [30]:
data = orders.merge(order_items, on="order_id")
data = data.merge(payments, on="order_id")
data = data.merge(reviews, on="order_id", how="left")
data = data.merge(products, on="product_id", how="left")

##Convert Dates

In [31]:
data['order_purchase_timestamp'] = pd.to_datetime(data['order_purchase_timestamp'])
data['order_delivered_customer_date'] = pd.to_datetime(
    data['order_delivered_customer_date'], errors='coerce'
)

##Deal with Missing values

In [32]:
#Check Missing Values Count
missing_values = data.isnull().sum()

print(missing_values)


order_id                              0
customer_id                           0
order_status                          0
order_purchase_timestamp              0
order_approved_at                    15
order_delivered_carrier_date       1254
order_delivered_customer_date      2588
order_estimated_delivery_date         0
order_item_id                         0
product_id                            0
seller_id                             0
shipping_limit_date                   0
price                                 0
freight_value                         0
payment_sequential                    0
payment_type                          0
payment_installments                  0
payment_value                         0
review_id                           978
review_score                        978
review_comment_title             104415
review_comment_message            68628
review_creation_date                978
review_answer_timestamp             978
product_category_name              1709


In [33]:
#Check Missing Percentage
missing_percentage = (data.isnull().sum() / len(data)) * 100

missing_df = pd.DataFrame({
    'Missing Values': data.isnull().sum(),
    'Percentage (%)': missing_percentage
}).sort_values(by='Percentage (%)', ascending=False)

print(missing_df)


                               Missing Values  Percentage (%)
review_comment_title                   104415       88.257669
review_comment_message                  68628       58.008402
order_delivered_customer_date            2588        2.187529
product_name_lenght                      1709        1.444547
product_category_name                    1709        1.444547
product_photos_qty                       1709        1.444547
product_description_lenght               1709        1.444547
order_delivered_carrier_date             1254        1.059954
review_id                                 978        0.826663
review_score                              978        0.826663
review_creation_date                      978        0.826663
review_answer_timestamp                   978        0.826663
product_length_cm                          20        0.016905
product_weight_g                           20        0.016905
product_width_cm                           20        0.016905
product_

In [34]:
#Check Total Missing Values
total_missing = data.isnull().sum().sum()
print("Total Missing Values in Dataset:", total_missing)

Total Missing Values in Dataset: 187728


In [35]:
#Focus on Important Columns Only
important_cols = [
    'price',
    'payment_value',
    'review_score',
    'review_comment_message',
    'product_category_name',
    'order_delivered_customer_date'
]

print(data[important_cols].isnull().sum())

price                                0
payment_value                        0
review_score                       978
review_comment_message           68628
product_category_name             1709
order_delivered_customer_date     2588
dtype: int64


In [36]:
#review_score feature Handeling
data['review_score'] = data['review_score'].fillna(data['review_score'].median())

In [37]:
#review_comment_message feature Handeling
# Fill missing text
data['review_comment_message'] = data['review_comment_message'].fillna("")

# Create indicator feature (VERY IMPORTANT for research)
data['has_review'] = data['review_comment_message'].apply(lambda x: 0 if x == "" else 1)

In [38]:
#product_category_name Handeling
data['product_category_name'] = data['product_category_name'].fillna("unknown")

In [39]:
#order_delivered_customer_date
# Convert to datetime
data['order_delivered_customer_date'] = pd.to_datetime(
    data['order_delivered_customer_date'], errors='coerce'
)

data['order_purchase_timestamp'] = pd.to_datetime(
    data['order_purchase_timestamp'], errors='coerce'
)

# Create delivery time
data['delivery_time'] = (
    data['order_delivered_customer_date'] - data['order_purchase_timestamp']
).dt.days

# Fill missing delivery time
data['delivery_time'] = data['delivery_time'].fillna(data['delivery_time'].median())

In [40]:
important_cols = [
    'price',
    'payment_value',
    'review_score',
    'review_comment_message',
    'product_category_name',
    'delivery_time'
]

print(data[important_cols].isnull().sum())

price                     0
payment_value             0
review_score              0
review_comment_message    0
product_category_name     0
delivery_time             0
dtype: int64


#Feature Engineerin

In [41]:
#Add Behavioral Feature capture Customer engagement and Sentiment strength
data['review_length'] = data['review_comment_message'].apply(len)

In [42]:
#create target feature
data['revenue'] = data['price']

In [43]:
#create Time feature
data['order_month'] = data['order_purchase_timestamp'].dt.to_period('M')

In [44]:
#create Sentiment Feature
from textblob import TextBlob

def get_sentiment(text):
    return TextBlob(text).sentiment.polarity

data['sentiment_score'] = data['review_comment_message'].apply(get_sentiment)

##Aggregate Monthly Dataset

In [45]:
monthly_data = data.groupby('order_month').agg({
    'revenue': 'sum',
    'order_id': 'nunique',
    'payment_value': 'sum',
    'delivery_time': 'mean',
    'review_score': 'mean',
    'sentiment_score': 'mean'
}).reset_index()

monthly_data.rename(columns={
    'order_id': 'num_orders'
}, inplace=True)

##Create Lag Features


In [46]:
monthly_data['lag_1'] = monthly_data['revenue'].shift(1)
monthly_data['lag_2'] = monthly_data['revenue'].shift(2)

monthly_data = monthly_data.dropna()

##Prepare Datasets for Each Model

In [47]:
#Financial Model
X_financial = monthly_data[
    ['num_orders', 'payment_value', 'delivery_time', 'lag_1', 'lag_2']
]

y = monthly_data['revenue']

In [ ]:
#Sentiment Model

y = monthly_data['revenue']

In [50]:
#Hybrid Model
X_hybrid = monthly_data[
    ['num_orders', 'payment_value', 'delivery_time',
     'lag_1', 'lag_2',
     'sentiment_score', 'review_score']
]

y = monthly_data['revenue']